# Actual GUI clean parity and bounded robustness

Integration diagnostic for frozen CLIP and selected M6 on **training-derived development only**. All new orchestration, integrity checks and comparisons are visible below; the actual published GUI predictor and unchanged WP5 corruption operators are deliberately imported as the components under test. This is not a new training implementation.

Default Run All loads metadata only. Historical packets have ranked IDs, not complete score matrices, so success means exact named ordered Top5 agreement and all available identity contracts—not full-score equality. No tolerance, altered scorer, new candidate universe or replacement reference is permitted.


## 1. Explicit controls and environment
Use the existing WSL Conda kernel. Source notebook flags stay off; authorized execution uses a separate saved copy.


In [ ]:
from pathlib import Path
import gc, hashlib, json, numbers, platform, sys, time, uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image
import torch, torchvision
import matplotlib.pyplot as plt

ROOT = Path("/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project")
def find_source_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents, ROOT):
        if ((candidate / "wp4/clip_predictor.py").is_file()
                and (candidate / "notebooks/gui_native_clean_parity.ipynb").is_file()):
            return candidate.resolve()
    raise FileNotFoundError("Open the notebook from this repository checkout")
WORKTREE = find_source_root()
IMAGE_DIR = ROOT / "workspace/datasets/DPD_Small_dataset"
if str(WORKTREE) not in sys.path: sys.path.insert(0, str(WORKTREE))
# This is an integration test of the ACTUAL GUI implementation, not a replacement encoder.
from wp4.service import Catalog
from wp4.clip_predictor import CLIPPredictor, catalog_contract, tensor_hash
from evaluation.robustness import NATIVE_SUITE, decode_rgb, pixel_digest, run_native_suite, suite_v2_fingerprint

LOAD_MODELS = False
RUN_CLEAN_PARITY = False
RUN_CORRUPTIONS = False
DEVICE = "cuda"
BATCH_SIZE = 8
EXPECTED_COUNTS = {"seen": 1238, "unseen": 1894}
DIRECT_BANK_SHA256 = "7f402c8dbebe821b1e1f2d799c9a0ce0e0d56a10a54aad563c6cd79d510df732"
MODEL_NAMES = ("clip_vit_l14_frozen", "clip_vit_l14_adapter")
OUTPUT_ROOT = ROOT / "workspace/experiments/gui-native-parity"
RUN_DIRECTORY = None
PREDICTORS = {}
CLEAN_RECEIPT = None
MODEL_PREFLIGHT = None

INPUTS = {name: Path(path) for name, path in {'source_csv': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/DPD_Small_train.csv', 'protocol': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/datasets/manifests/wp1_reconciled_protocol.json', 'identity_manifest': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs/WP2_WP3_IDENTIFIED_HANDOFF_20261107/identity_manifest.json', 'base': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/pretrained_weights/clip/ViT-L-14.pt', 'adapter': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/experiments/wp3b-m6-one-epoch/20261105T064914Z_f2f8ff712a41/seed_45_bounded/adapter_epoch_001_e84ba1dd.pt', 'cache': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/experiments/wp3b-m4/feature_cache/20261104T180329Z_69b6bc6626a7.pt', 'clip_vit_l14_frozen_seen': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs/WP2_WP3_IDENTIFIED_HANDOFF_20261107/packets/clip_vit_l14_frozen_seen.json', 'clip_vit_l14_frozen_unseen': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs/WP2_WP3_IDENTIFIED_HANDOFF_20261107/packets/clip_vit_l14_frozen_unseen.json', 'clip_vit_l14_adapter_seen': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs/WP2_WP3_IDENTIFIED_HANDOFF_20261107/packets/clip_vit_l14_adapter_seen.json', 'clip_vit_l14_adapter_unseen': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/handoffs/WP2_WP3_IDENTIFIED_HANDOFF_20261107/packets/clip_vit_l14_adapter_unseen.json', 'source:wp4/clip_predictor.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/wp4/clip_predictor.py', 'source:wp4/service.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/wp4/service.py', 'source:shared_protocol.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/shared_protocol.py', 'source:data_utils.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/data_utils.py', 'source:evaluation/robustness.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/evaluation/robustness.py', 'source:evaluation/reporting.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/evaluation/reporting.py', 'source:evaluation/metrics.py': '/home/user1/COS30082 Applied Machine Learning/00 Assigments/AS02 ML Project/workspace/worktrees/gui-native-parity/evaluation/metrics.py'}.items()}
EXPECTED_SHA256 = {'source_csv': '9f1b729ab08f120c9e8d3a5681aa1ccbdf1d9210f9aaa6143959363dcb106a7d', 'protocol': '9efc2265f8b30f83408954fa936f55bbd805de71f8db477100a690a2c1d04c09', 'identity_manifest': '206e4846a149ea4edce76893358702fca4e0133b391c4f32949fa5d97b0a0a75', 'base': 'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836', 'adapter': '8835a18b9eb933ae224bd17866f18bc56406e24d8612743f4e7c88848a8135cc', 'cache': '9dbb07ae03ba38d003f5e4c610498a9a8c5ecb57475ec2f04f8e8481deeaa38d', 'clip_vit_l14_frozen_seen': '378554f85c3322bd08437fc7edacca9961f63ff859bb6cf09c3800a618d95c85', 'clip_vit_l14_frozen_unseen': 'b83cd174eb05f5f81aed387741de2af6ecd12efcaf474e400c88bd59f70ca50f', 'clip_vit_l14_adapter_seen': '22d5f5a133c0e9dc57dae06f92fb383b8e47100b47e2289017846defda1e0e0f', 'clip_vit_l14_adapter_unseen': '9a3fb504695be2aa2eb37beeb5c4ab09c7dc6cc0907a54e87de6d36c255768d6', 'source:wp4/clip_predictor.py': '2e9066c33e858b597208fea7f636650729dea675803324ce626a4594fe89e986', 'source:wp4/service.py': '815960cacbc82bf4e7a40e265bee57b6ab3ce73a0359645c36c724e7b1b901a0', 'source:shared_protocol.py': 'b38885f646dbdf9dde51326620d82334233a2d618371ac06a5c1d8ecdd0f00d3', 'source:data_utils.py': 'f147c42433b7feee831833f8a060e14517e2a8a08a62114cae825327b0683eff', 'source:evaluation/robustness.py': 'dd5b5ec3e20d408613b95fc1dbf48d13c730efeec899d1d40db29a67a7bf3613', 'source:evaluation/reporting.py': 'b07b867be4d2fa648f7193656830c5c07701fd14c5ffc43648b3fea88752b4f0', 'source:evaluation/metrics.py': 'e301281bdd3dfc10f3b2cdd94bf0d1d5b7b77415334a6fbc66df216cd2127558'}
print({"python": platform.python_version(), "torch": str(torch.__version__),
       "torchvision": str(torchvision.__version__), "cuda_available": torch.cuda.is_available(),
       "worktree": str(WORKTREE), "load_models": LOAD_MODELS,
       "run_clean_parity": RUN_CLEAN_PARITY, "run_corruptions": RUN_CORRUPTIONS})

# Private assets stay in the explicitly configured canonical workspace.
# Predictor and evaluator source always come from THIS checkout.
for name in INPUTS:
    if name.startswith("source:"): INPUTS[name] = WORKTREE / name.split(":",1)[1]


## 2. File integrity and historical RGB decoder
All opened images must be among3132approved development filenames, pass their current byte hash, and use the historical untransposed RGB decode. Official metadata/images are outside this workflow.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""): digest.update(block)
    return digest.hexdigest()


def verify_input(name):
    path = INPUTS[name]
    if not path.is_file() or sha256_file(path) != EXPECTED_SHA256[name]:
        raise ValueError("Pinned file missing or changed: " + name)
    return path


def new_run_directory():
    parent = OUTPUT_ROOT
    parent.mkdir(parents=True, exist_ok=True)
    target = parent / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:12])
    target.mkdir(exist_ok=False)
    return target


def write_json_exclusive(path, payload):
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(payload, handle, sort_keys=True, indent=2, allow_nan=False)
        handle.write("\n")


def historical_rgb(name):
    if name not in ALLOWED_NAMES: raise ValueError("Image is outside frozen development folds")
    path = (IMAGE_DIR / name).resolve()
    if path.parent != IMAGE_DIR.resolve(): raise ValueError("Image path escaped the flat dataset")
    # Verify the CURRENT bytes, not only a saved hash ledger.
    if sha256_file(path) != APPROVED_IMAGE_HASHES[name]: raise ValueError("Image bytes changed: " + name)
    with Image.open(path) as source:
        image = source.convert("RGB").copy()  # deliberately no EXIF transpose
    CLEAN_PIXEL_DIGESTS[name] = pixel_digest(np.asarray(image, dtype=np.uint8))
    return image


def display_progress(model, fold, current, total):
    if current == total or current % 256 == 0:
        print(f"{model} / {fold}: {current}/{total}", flush=True)


## 3. Metadata and exact candidate/fold identity
The reconciled protocol is supplied explicitly. No regeneration of the historical split.


In [ ]:
# Metadata-only preflight: only the ORIGINAL TRAINING source and named development packets.
for key in INPUTS:
    if key not in ("base", "adapter", "cache"): verify_input(key)
PROTOCOL = json.loads(INPUTS["protocol"].read_text())
CATALOG = Catalog(INPUTS["protocol"], INPUTS["source_csv"], IMAGE_DIR)
CONTRACT = catalog_contract(CATALOG)
CANDIDATE_IDS = CONTRACT["ids"]
FOLDS = {"seen": PROTOCOL["seen_validation_images"], "unseen": PROTOCOL["unseen_validation_images"]}
MANIFEST = json.loads(INPUTS["identity_manifest"].read_text())
TRUTH_LOOKUP = {row["sample_id"]: row["truth_id"] for row in MANIFEST["samples"]}
TRUTHS = {fold: [TRUTH_LOOKUP[name] for name in names] for fold, names in FOLDS.items()}
ALLOWED_NAMES = frozenset(name for names in FOLDS.values() for name in names)
assert len(ALLOWED_NAMES) == 3132
assert len(PROTOCOL["fitting_images"]) == 4868 and not ALLOWED_NAMES.intersection(PROTOCOL["fitting_images"])
REFERENCES = {model: {fold: json.loads(INPUTS[model + "_" + fold].read_text()) for fold in FOLDS} for model in MODEL_NAMES}
for model in MODEL_NAMES:
    for fold, names in FOLDS.items():
        packet = REFERENCES[model][fold]
        assert packet["image_names"] == names and packet["ground_truth_ids"] == TRUTHS[fold]
        assert packet["candidate_ids"] == CANDIDATE_IDS
        assert packet["prompt_sha256"] == "753672c7964bcb20a1346145f21a72b40a19f55590a006b5783823f23c13f537"
print(pd.DataFrame([{"fold": fold, "count": len(names), "candidates": len(CANDIDATE_IDS)} for fold,names in FOLDS.items()]))
print("Rank-only references: no historical full-score equality is claimed.")


## 4. Complete clean gate and strict prediction contract
Every named reference is checked before image inference. Both entire folds for both models must finish successfully before the guard can permit any corruption.


In [ ]:
def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def strict_ids(value, label):
    """Reject floats, strings and booleans before any integer conversion."""
    if isinstance(value, torch.Tensor):
        if value.dtype not in (torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64):
            raise ValueError(label + ": integer tensor required")
        array = value.detach().cpu().numpy()
    elif isinstance(value, np.ndarray):
        if value.dtype.kind not in "iu":
            raise ValueError(label + ": integer array required")
        array = value
    else:
        def check(item):
            if isinstance(item, (list, tuple)):
                for child in item: check(child)
            elif isinstance(item, bool) or not isinstance(item, numbers.Integral):
                raise ValueError(label + ": integer IDs required, without coercion")
        check(value)
        try:
            array = np.asarray(value)
        except (TypeError, ValueError) as error:
            raise ValueError(label + ": rectangular integer IDs required") from error
    if array.dtype.kind not in "iu" or array.size == 0 or np.any(array < 0) or np.any(array > np.iinfo(np.int64).max):
        raise ValueError(label + ": nonempty nonnegative int64-compatible IDs required")
    return array.astype(np.int64, copy=True)


def checked_output(output, ids, count):
    declared = strict_ids(output["candidate_ids"], "predictor candidate IDs")
    if declared.shape != ids.shape or not np.array_equal(declared, ids):
        raise ValueError("Predictor candidate columns changed")
    scores = output["scores"]
    if isinstance(scores, torch.Tensor): scores = scores.detach().cpu().numpy()
    scores = np.asarray(scores)
    if scores.dtype != np.float64 or scores.shape != (count, len(ids)) or not np.isfinite(scores).all():
        raise ValueError("Expected finite FP64 [batch, candidates] scores")
    if output.get("score_semantics") != "direct_float64_clip_cosine_similarity":
        raise ValueError("Unexpected predictor scoring contract")
    k = min(5, len(ids))
    # Ranking only; actual cosine scoring remains the unchanged real GUI implementation.
    order = torch.from_numpy(np.array(scores, copy=True)).argsort(dim=1, descending=True, stable=True)[:, :k].numpy()
    expected = ids[order]
    returned = strict_ids(output["topk_composition_ids"], "predictor ranked IDs")
    if returned.shape != expected.shape or not np.array_equal(returned, expected):
        raise ValueError("Predictor ranking disagrees with stable canonical scores")
    return scores.copy(), returned.copy()


def clean_task_hash(models, folds, truths, candidate_ids, counts):
    return canonical_hash({"schema":"actual-gui-clean-task-v1", "models":sorted(models),
        "folds":folds, "truths":truths, "candidate_ids":candidate_ids, "counts":counts})


def persist_clean_run(operation, directory):
    """Record late exceptions without claiming a completed pass or enabling corruptions."""
    try:
        result = operation()
    except Exception as error:
        write_json_exclusive(Path(directory) / "clean_failure.json", {
            "schema":"actual-gui-clean-failure-v1", "passed":False, "corrupted_calls":0,
            "error_type":type(error).__name__, "message":str(error),
            "processed_rows":"not established; inspect saved notebook progress"})
        raise
    write_json_exclusive(Path(directory) / "clean_receipt.json", result)
    return result


def run_clean_gate(predictors, folds, truths, candidate_ids, references, load_rgb,
                   *, batch_size=8, expected_counts=None, progress=None):
    """Complete EVERY model/fold clean pass before issuing any pass receipt.

    Uses real predictor inference in the owner run; injected tiny predictors only
    exercise orchestration in CPU tests. References are immutable named packets.
    """
    ids = strict_ids(candidate_ids, "candidate IDs")
    if ids.ndim != 1 or ids.tolist() != sorted(set(ids.tolist())):
        raise ValueError("Candidates must be unique ascending original global IDs")
    if type(batch_size) is not int or batch_size < 1 or not predictors:
        raise ValueError("Positive batch size and predictors required")
    if set(folds) != {"seen", "unseen"} or set(truths) != set(folds):
        raise ValueError("Both complete development folds required")
    expected_counts = expected_counts or {"seen": 1238, "unseen": 1894}
    if set(expected_counts) != set(folds) or set(references) != set(predictors):
        raise ValueError("Missing fold or model reference")
    all_names = []
    true_arrays = {}
    for fold, names in folds.items():
        count = expected_counts[fold]
        if type(count) is not int or count < 1 or len(names) != count or len(set(names)) != count:
            raise ValueError("Fold length or filename uniqueness mismatch")
        if any(not isinstance(name, str) or not name or name in (".", "..") or any(c in name for c in "/\\") for name in names):
            raise ValueError("Only approved flat filenames may be read")
        truth = strict_ids(truths[fold], "ground-truth IDs")
        if truth.shape != (count,) or not np.isin(truth, ids).all():
            raise ValueError("Truth shape or candidate membership mismatch")
        true_arrays[fold] = truth
        all_names.extend(names)
    if len(set(all_names)) != len(all_names):
        raise ValueError("Development folds overlap")
    # Validate every reference BEFORE the first model call or image decode.
    checked = {}
    for model, by_fold in references.items():
        if set(by_fold) != set(folds): raise ValueError("Incomplete model reference")
        checked[model] = {}
        for fold, packet in by_fold.items():
            ref_ids = strict_ids(packet["candidate_ids"], "reference candidate IDs")
            ref_truth = strict_ids(packet["ground_truth_ids"], "reference truth IDs")
            ranked = strict_ids(packet["ranked_ids"], "reference ranked IDs")
            k = min(5, len(ids))
            if (ref_ids.shape != ids.shape or not np.array_equal(ref_ids, ids)
                    or packet["image_names"] != folds[fold]
                    or ref_truth.shape != true_arrays[fold].shape or not np.array_equal(ref_truth, true_arrays[fold])
                    or type(packet["available_k"]) is not int or packet["available_k"] != k
                    or ranked.shape != (len(folds[fold]), k) or not np.isin(ranked, ids).all()
                    or any(len(set(row)) != k for row in ranked.tolist())):
                raise ValueError("Reference identity, truths or complete Top5 contract mismatch")
            checked[model][fold] = ranked
    started = time.perf_counter()
    result = {"schema": "actual-gui-clean-gate-v1", "passed": False,
              "expected_rows": sum(expected_counts.values()) * len(predictors),
              "task_sha256": clean_task_hash(list(predictors), folds, truths, ids.tolist(), expected_counts),
              "processed_rows": 0, "mismatch_rows": 0, "mismatches": [],
              "model_fold_results": {}, "corrupted_calls": 0}
    for model, predictor in predictors.items():
        result["model_fold_results"][model] = {}
        for fold, names in folds.items():
            ranks = []
            for start in range(0, len(names), batch_size):
                batch_names = names[start:start + batch_size]
                images = [load_rgb(name) for name in batch_names]
                _, ranked = checked_output(predictor.predict(images, topk=min(5, len(ids))), ids, len(images))
                ranks.extend(ranked.tolist())
                result["processed_rows"] += len(images)
                if progress is not None: progress(model, fold, min(start + batch_size, len(names)), len(names))
            actual = np.asarray(ranks, dtype=np.int64)
            wanted = checked[model][fold]
            wrong = np.flatnonzero(np.any(actual != wanted, axis=1))
            result["mismatch_rows"] += len(wrong)
            for row in wrong:
                result["mismatches"].append({"model": model, "fold": fold, "filename": names[row],
                    "truth_id": int(true_arrays[fold][row]), "reference_ranked_ids": wanted[row].tolist(),
                    "actual_ranked_ids": actual[row].tolist(),
                    "top1_changed": bool(actual[row, 0] != wanted[row, 0])})
            truth = true_arrays[fold]
            result["model_fold_results"][model][fold] = {
                "count": len(names), "ordered_top5_mismatches": len(wrong),
                "actual_top1_hits": int(np.sum(actual[:, 0] == truth)),
                "actual_top5_hits": int(np.sum(np.any(actual == truth[:, None], axis=1))),
                "reference_top1_hits": int(np.sum(wanted[:, 0] == truth)),
                "reference_top5_hits": int(np.sum(np.any(wanted == truth[:, None], axis=1))),
                "image_names": list(names), "ranked_ids": ranks}
    result["elapsed_seconds"] = time.perf_counter() - started
    result["passed"] = result["processed_rows"] == result["expected_rows"] and result["mismatch_rows"] == 0
    result["receipt_sha256"] = canonical_hash(result)
    return result


def run_guarded_corruptions(clean_receipt, enabled, runner, *, expected_task_sha256=None):
    """No corruption before the full unchanged clean receipt passes and user enables it."""
    if type(enabled) is not bool: raise ValueError("Explicit boolean corruption control required")
    if not enabled: return None
    if not isinstance(clean_receipt, dict): raise ValueError("Complete clean receipt required")
    if not isinstance(expected_task_sha256, str) or len(expected_task_sha256) != 64:
        raise ValueError("The independently declared complete task hash is required")
    payload = {key: value for key, value in clean_receipt.items() if key != "receipt_sha256"}
    if (clean_receipt.get("schema") != "actual-gui-clean-gate-v1" or clean_receipt.get("passed") is not True
            or clean_receipt.get("processed_rows") != clean_receipt.get("expected_rows")
            or clean_receipt.get("mismatch_rows") != 0 or clean_receipt.get("corrupted_calls") != 0
            or clean_receipt.get("receipt_sha256") != canonical_hash(payload)
            or clean_receipt.get("task_sha256") != expected_task_sha256):
        raise ValueError("Full clean parity failed, incomplete or mutated: no corrupted calls allowed")
    return runner()


## 5. Explicit model/native-bank preflight
CUDA native FP16 encoder, image batch8, text batch32; saved M6 FP32 adapter and FP64 scoring. Original unrecorded backend flags are not guessed or changed. Any direct-bank difference stops the gate before image inference.


In [ ]:
EXPECTED_TASK_SHA256 = clean_task_hash(list(MODEL_NAMES), FOLDS, TRUTHS, CANDIDATE_IDS, EXPECTED_COUNTS)
if LOAD_MODELS:
    if PREDICTORS or RUN_DIRECTORY is not None: raise RuntimeError("Use a fresh kernel; do not overwrite or repeat model initialization")
    if DEVICE != "cuda" or not torch.cuda.is_available(): raise ValueError("Historical parity requires the recorded CUDA path")
    if (platform.python_version(), str(torch.__version__), str(torchvision.__version__)) != ("3.10.20", "2.5.1+cu121", "0.20.1+cu121"):
        raise ValueError("Use the approved recorded Katana environment")
    for name in ("base", "adapter", "cache"): verify_input(name)
    RUN_DIRECTORY = new_run_directory()
    cache = torch.load(INPUTS["cache"], map_location="cpu", weights_only=True)
    if cache["direct_sha256"] != DIRECT_BANK_SHA256 or tensor_hash(cache["direct"]) != DIRECT_BANK_SHA256:
        raise ValueError("Reference direct text bank changed")
    APPROVED_IMAGE_HASHES = {name: cache["image_hashes"][name] for name in ALLOWED_NAMES}
    for fold, key in (("seen", "val_seen"), ("unseen", "val_unseen")):
        assert cache["folds"][key]["filenames"] == FOLDS[fold]
    CLEAN_PIXEL_DIGESTS = {}
    MODEL_PREFLIGHT = {"passed": False, "models": {}, "environment": {
        "python": platform.python_version(), "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
        "pillow": __import__("PIL").__version__, "device": DEVICE,
        "gpu": torch.cuda.get_device_name(), "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
        "cuda_matmul_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_tf32": torch.backends.cudnn.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark, "deterministic_algorithms": torch.are_deterministic_algorithms_enabled()},
        "input_hashes": EXPECTED_SHA256, "decode": "untransposed_RGB", "image_batch_size": BATCH_SIZE,
        "text_batch_size": 32, "direct_bank_sha256": DIRECT_BANK_SHA256}
    try:
        for model in MODEL_NAMES:
            adapter_path = INPUTS["adapter"] if model == "clip_vit_l14_adapter" else None
            predictor = CLIPPredictor(CATALOG, INPUTS["base"], adapter_path, device=DEVICE)
            bank_hash = tensor_hash(predictor.direct)
            MODEL_PREFLIGHT["models"][model] = {"direct_bank_sha256": bank_hash, "provenance": predictor.provenance,
                "encoder_dtype": str(next(predictor.model.parameters()).dtype)}
            if bank_hash != DIRECT_BANK_SHA256: raise ValueError("Native text bank differs from historical saved bank: " + model)
            if predictor.model.training or any(p.requires_grad for p in predictor.model.parameters()): raise ValueError("CLIP is not frozen")
            PREDICTORS[model] = predictor
        MODEL_PREFLIGHT["passed"] = True
    except Exception as error:
        MODEL_PREFLIGHT["failure"] = {"type": type(error).__name__, "message": str(error)}
        print("STOP: model/native-bank preflight failed; no image or corruption inference is authorized by this receipt.", MODEL_PREFLIGHT["failure"])
    write_json_exclusive(RUN_DIRECTORY / "model_preflight.json", MODEL_PREFLIGHT)
    print({"run_directory": str(RUN_DIRECTORY), "model_preflight_passed": MODEL_PREFLIGHT["passed"]})
    del cache
else:
    print("LOAD_MODELS=False: no weights, cache or image is loaded; no model construction.")


## 6. Full clean development inference
1238seen+1894unseen rows per model. Record mismatches without rewriting historical results. No training, prompt tuning, checkpoint reselection or official tests.


In [ ]:
if RUN_CLEAN_PARITY:
    if CLEAN_RECEIPT is not None: raise RuntimeError("Clean pass already recorded; use a new explicitly declared run")
    if not LOAD_MODELS or not MODEL_PREFLIGHT or not MODEL_PREFLIGHT["passed"]:
        print("CLEAN BLOCKED: verified real predictors/native text bank are required. Corruptions remain blocked.")
    else:
        torch.cuda.reset_peak_memory_stats()
        CLEAN_RECEIPT = persist_clean_run(lambda: run_clean_gate(PREDICTORS, FOLDS, TRUTHS, CANDIDATE_IDS, REFERENCES, historical_rgb,
            batch_size=BATCH_SIZE, expected_counts=EXPECTED_COUNTS, progress=display_progress), RUN_DIRECTORY)
        write_json_exclusive(RUN_DIRECTORY / "clean_pixel_hashes.json", CLEAN_PIXEL_DIGESTS)
        clean_summary = {"passed": CLEAN_RECEIPT["passed"], "processed_rows": CLEAN_RECEIPT["processed_rows"],
            "mismatch_rows": CLEAN_RECEIPT["mismatch_rows"], "elapsed_seconds": CLEAN_RECEIPT["elapsed_seconds"],
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(), "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
            "by_model_fold": [{"model": model,"fold": fold, **{k:v for k,v in row.items() if k not in ("image_names","ranked_ids")}}
                for model,folds in CLEAN_RECEIPT["model_fold_results"].items() for fold,row in folds.items()]}
        write_json_exclusive(RUN_DIRECTORY / "clean_summary.json", clean_summary)
        print(pd.DataFrame(clean_summary["by_model_fold"]))
        print({key:value for key,value in clean_summary.items() if key != "by_model_fold"})
        if not CLEAN_RECEIPT["passed"]:
            print("CLEAN GATE FAILED: do not run corruption sweeps, replace references or introduce tolerances.")
            print(pd.DataFrame(CLEAN_RECEIPT["mismatches"][:12]))
else:
    print("RUN_CLEAN_PARITY=False: no real image inference.")


## 7. Separately guarded bounded robustness
Predeclared first4filenames/composition/fold, severity1 of the published WP5v2 suite. Subsampled sensitivity only, not full-fold accuracy. Batch boundaries in this subset differ from historical full-fold batches and must be disclosed. The full clean gate is mandatory first; no corruptions occur on failure.


In [ ]:
# Predeclared bounded sensitivity probe ONLY after complete clean parity.
# First4 image names per global composition in each fold, sorted by filename,
# independent of correctness/model outputs. All65unseen compositions retained.
ROBUSTNESS_NAMES = {}
for fold,names in FOLDS.items():
    by_class = {}
    for name in sorted(names): by_class.setdefault(TRUTH_LOOKUP[name], []).append(name)
    selected = {name for values in by_class.values() for name in values[:4]}
    ROBUSTNESS_NAMES[fold] = [name for name in names if name in selected]
# Same published severity1 settings; no tuning after viewing perturbation results.
ROBUSTNESS_SUITE = tuple(p for p in NATIVE_SUITE if p.family == "clean" or p.severity == 1)
print({"bounded_robustness_counts": {f:len(n) for f,n in ROBUSTNESS_NAMES.items()},
       "suite_sha256": suite_v2_fingerprint(ROBUSTNESS_SUITE), "conditions": [p.name for p in ROBUSTNESS_SUITE]})

def run_bounded_robustness():
    results = {}
    for fold,names in ROBUSTNESS_NAMES.items():
        # Original native image arrays; no EXIF correction and no model-specific corruption.
        def load_name(name): return np.asarray(historical_rgb(name), dtype=np.uint8).copy()
        results[fold] = run_native_suite(PREDICTORS, names, load=load_name, suite=ROBUSTNESS_SUITE,
            batch_sizes={model:BATCH_SIZE for model in MODEL_NAMES},
            outputs={model:("scores", "topk_composition_ids") for model in MODEL_NAMES}, workers=2,
            progress=lambda n,total: print(f"robustness {fold}: {n}/{total}", flush=True) if n == total or n%256==0 else None)
        arrays = {f"{model}__{condition}__{field}":value for model in MODEL_NAMES
                  for condition,row in results[fold][model].items() for field,value in row.items()}
        with (RUN_DIRECTORY / ("robustness_"+fold+".npz")).open("xb") as handle: np.savez_compressed(handle, **arrays)
        write_json_exclusive(RUN_DIRECTORY / ("robustness_"+fold+"_pixels.json"), results[fold]["pixel_sha256"])
    rows = []
    for model in MODEL_NAMES:
        for condition in [p.name for p in ROBUSTNESS_SUITE]:
            pair = {}
            for fold,names in ROBUSTNESS_NAMES.items():
                scores = results[fold][model][condition]["scores"]
                ranks = np.asarray(CANDIDATE_IDS)[torch.from_numpy(scores).argsort(dim=1,descending=True,stable=True)[:,:5].numpy()]
                truth = np.array([TRUTH_LOOKUP[name] for name in names])
                pair[fold] = {"count":len(names), "top1":float(np.mean(ranks[:,0]==truth)), "top5":float(np.mean((ranks==truth[:,None]).any(1)))}
            s,u = pair["seen"],pair["unseen"]
            h = lambda a,b: 0. if a+b==0 else 2*a*b/(a+b)
            rows.append({"model":model,"condition":condition,"seen_count":s["count"],"unseen_count":u["count"],
                "seen_top1":s["top1"],"unseen_top1":u["top1"],"H1":h(s["top1"],u["top1"]),
                "seen_top5":s["top5"],"unseen_top5":u["top5"],"H5":h(s["top5"],u["top5"])})
    write_json_exclusive(RUN_DIRECTORY / "bounded_robustness_summary.json", rows)
    return rows

if RUN_CORRUPTIONS:
    ROBUSTNESS_ROWS = run_guarded_corruptions(CLEAN_RECEIPT, True, run_bounded_robustness, expected_task_sha256=EXPECTED_TASK_SHA256)
    print(pd.DataFrame(ROBUSTNESS_ROWS))
else:
    print("RUN_CORRUPTIONS=False: no corrupted image prediction. No robustness claim.")


## 8. Results and interpretation
A failed exact gate establishes a reproducibility mismatch, not proof of worse predictive accuracy. Investigate bytes, decode, native bank, runtime/batches and preprocessing before changing anything. GUI EXIF-corrected CPU/batch1 behavior remains a separate labelled path.


In [ ]:
if CLEAN_RECEIPT is not None:
    table = pd.DataFrame([{"model":model,"fold":fold,"rank_mismatches":row["ordered_top5_mismatches"],"count":row["count"]}
        for model,folds in CLEAN_RECEIPT["model_fold_results"].items() for fold,row in folds.items()])
    display(table)
    fig, ax = plt.subplots(figsize=(8,3))
    ax.bar([row["model"].replace("clip_vit_l14_","")+" / "+row["fold"] for row in table.to_dict("records")],table["rank_mismatches"])
    ax.set_ylabel("Rows with ordered Top5 mismatch");ax.set_title("Actual GUI predictor versus historical named development packets")
    ax.tick_params(axis="x",labelrotation=20);fig.tight_layout();plt.show()
else:
    print("No clean predictions yet. Do not fabricate metric/parity charts.")
if RUN_DIRECTORY is not None:
    print("Actual saved evidence:", RUN_DIRECTORY)
print("Primary M6, historical packets and official evaluation remain unchanged.")

if RUN_CORRUPTIONS and "ROBUSTNESS_ROWS" in globals():
    frame = pd.DataFrame(ROBUSTNESS_ROWS)
    fig, ax = plt.subplots(figsize=(9, 4))
    conditions = [p.name for p in ROBUSTNESS_SUITE]
    for model, label in (("clip_vit_l14_frozen", "Frozen CLIP"), ("clip_vit_l14_adapter", "Selected M6")):
        rows = frame[frame.model == model].set_index("condition").loc[conditions]
        ax.plot(conditions, rows["H1"] * 100, marker="o", label=label)
    ax.set_ylabel("H@1 (%)"); ax.set_title("Native sensitivity: 533 seen + 218 unseen; class-capped subset")
    ax.tick_params(axis="x", labelrotation=30); ax.grid(axis="y", alpha=.25); ax.legend(); fig.tight_layout(); plt.show()


## Owner/authorized execution
1. Open a new execution copy in VS Code WSL; selectas04_yolo_katana.
2. EnableLOAD_MODELS andRUN_CLEAN_PARITY only; leaveRUN_CORRUPTIONS=False. Run All and save actual outputs.
3. If preflight or any row fails, preserve the receipts and leave corruptions disabled. Never replace historical ranks to force success.
4. Only on a complete clean pass and explicit execution, enableRUN_CORRUPTIONS and run Section7 once; exclusive outputs prevent overwrites.
5. Close the kernel when finished. Primary M6 and all historical results are unchanged.
